# Enhancing AI Agent Tool-Calling Accuracy Through Fine-Tuning


## Agenda
![Title Diagram](./img/agenda.png)

## Demo Goal: Improving the Zava Retail Agent

**Zava** is a fictional retail brand with an AI agent powered by Microsoft Foundry and a Retail MCP Server.

**What works:** Base models handle simple single-tool calls (e.g., "show me products") ✅  
**What fails:** Complex multi-turn scenarios requiring tool chaining and parameter propagation ❌

**This notebook demonstrates:**
1. Synthetic data generation using Microsoft Foundry
2. Supervised fine-tuning (SFT) for tool-calling accuracy
3. Python-based evaluation with custom graders
4. Measuring improvements across model sizes

## 1. Architecture Overview

### The Zava Agent System

**Microsoft Foundry Agent Service** (orchestrator)
- Interprets customer intent
- Selects appropriate tools
- Constructs structured tool-call arguments
- Produces customer-friendly responses

**Retail MCP Server** (tool provider)  
Exposes 9 tools across 4 categories:

```
User Lookup
 ├─ find_user_id_by_email
 ├─ find_user_id_by_name_zip
 └─ get_user_details

Orders
 ├─ get_order_details
 ├─ cancel_pending_order
 └─ exchange_delivered_order_items

Products
 ├─ list_products
 └─ get_product_details

Customer Profile
 └─ update_address
```

### Request Flow

```
Customer Message
       ↓
Agent (LLM + Policies + Tool Schemas)
       ↓
MCP Server (executes tool)
       ↓
Tool Output (JSON) → Agent Reasoning
       ↓
Final Response
```

### Architecture Diagram

<img src="img/architecture.png" alt="Zava Architecture" width="600">

### What Fine-Tuning Improves

- **Tool chaining** - Correct sequence of tool calls
- **Parameter propagation** - Carrying IDs and data across turns
- **Policy alignment** - Enforcing business rules
- **Multi-turn consistency** - Maintaining state over long interactions

## 2. Prerequisites & Environment Setup

### Azure Resources Required

✅ Azure AI Foundry Hub & Project  
✅ Azure OpenAI deployments: `gpt-4.1`, `gpt-4.1-mini`, `gpt-4.1-nano`  
✅ Azure AI Search with vector index populated with product embeddings  
✅ Retail MCP Server running and accessible  


See [infra/README.md](../src/00-setup.md) for detailed setup instructions.


### Test MCP Server Connectivity

Verify that your MCP server is reachable and the agent can access it.

In [ ]:
# Quick connectivity test using the MCP test utility
import sys
sys.path.append('tools')

from test_mcp_connectivity import MCPConnectivityTester, quick_test
from dotenv import load_dotenv
import os

load_dotenv()

# Get MCP server URL from environment or use default
mcp_url = os.getenv("MCP_SERVER_URL")

mcp_url = mcp_url.rstrip('/mcp')
print(f"🔍 Testing MCP Server\n")
print(f"🌐 MCP Server URL: {mcp_url}\n")

import time
time.sleep(60)

# Quick test first
if quick_test(mcp_url):
    print("✅ Server is reachable!\n")
    
    # Run full test suite with notebook-friendly formatting
    tester = MCPConnectivityTester(mcp_url)
    tester.run_all_tests(notebook_mode=True)
else:
    print("❌ Server is not reachable. Please check the URL and your network connection.")


### Initialize Retail Agent Tester

The `RetailAgentTester` provides helper methods to interact with your agent and MCP server.

In [ ]:
# Testing Retail Agent with Microsoft Foundry
import sys
sys.path.append('tools')

from test_retail_agent import RetailAgentTester
from dotenv import load_dotenv
import os

load_dotenv()

print("🤖 Testing Microsoft Foundry Retail Agent\n")

# Initialize the tester
tester = RetailAgentTester()

# Run all tests with notebook-friendly formatting
results = tester.run_all_tests(notebook_mode=True)

## 4. Agent Creation with Tools & Policies

### System Instructions & Business Policy

The agent receives:
1. **Identity**: "You are a helpful customer service agent for Zava..."
2. **Capabilities**: Access to user lookup, orders, products, profile updates via MCP
3. **Business Rules**: Encoded in the policy document

**Key Policy Rules:**

```markdown
EXCHANGE_POLICY
- Delivered orders: Exchange ONLY, no cancellations
- Pending orders: Cancel or exchange allowed
- Must use cancel_pending_order() or exchange_delivered_order_items()

USER_IDENTIFICATION_POLICY
- ALWAYS look up user_id FIRST before any order/profile operations
- Use find_user_id_by_email() or find_user_id_by_name_zip()
- Never proceed without valid user_id

INFORMATION_DISCLOSURE_POLICY
- Share order/user info ONLY after successful identification
- No speculation on unavailable data
```

These policies are critical - base models often violate them without fine-tuning.

### OpenAPI Specification with Policy Annotations

The MCP server exposes its tools via an OpenAPI spec enriched with business policies:

**Structure:**
- **paths**: Tool endpoints (e.g., `/find_user_id_by_name_zip`)
- **x-policy**: Policy annotations per tool
- **parameters**: Required/optional fields with validation rules

**Example Tool Definition:**

```json
{
  "paths": {
    "/find_user_id_by_name_zip": {
      "post": {
        "summary": "Find user ID using name and zip code",
        "x-policy": "ALWAYS use this before operations requiring user_id",
        "parameters": {
          "name": {"type": "string", "required": true},
          "zip": {"type": "string", "required": true}
        }
      }
    }
  }
}
```

The `x-policy` fields reinforce behavioral constraints that must be learned through fine-tuning.

### Baseline Agent Behavior (Pre-Fine-Tuning)

**Test Scenario:**  
Customer: *"Hi, I'm Noah, zip code 80279, and I want to update my address."*

**Expected Flow:**
1. ✅ Call `find_user_id_by_name_zip(name="Noah", zip="80279")`
2. ✅ Get `user_id` from response
3. ✅ Call `update_address(user_id=<id>, new_address=<address>)`

**Actual Base Model Behavior:**

❌ **Problem 1: Skips User Lookup**  
Calls `update_address()` immediately without `user_id`

❌ **Problem 2: Hallucinates Parameters**  
Invents fake `user_id` instead of calling lookup tool

❌ **Problem 3: Violates Policy**  
Ignores `USER_IDENTIFICATION_POLICY` requiring lookup first

**Why It Fails:**
- Base models lack training on complex tool chaining patterns
- They don't internalize policy constraints from system prompts alone
- Parameter propagation across tools requires explicit fine-tuning examples

**Solution:** Supervised fine-tuning with tool-calling demonstrations

### Create Microsoft Foundry Agent

Initialize the agent with:
- Model deployment (gpt-4.1)
- System instructions + business policies
- OpenAPI tool schema from MCP server
- Azure AI Search integration

In [ ]:
SYSTEM_PROMPT = """
# Retail agent policy

As a retail agent, you can help users:

- **cancel or modify pending orders**
- **return or exchange delivered orders**
- **modify their default user address**
- **provide information about their own profile, orders, and related products**

At the beginning of the conversation, you have to authenticate the user identity by locating their user id via email, or via name + zip code. This has to be done even when the user already provides the user id.

Once the user has been authenticated, you can provide the user with information about order, product, profile information, e.g. help the user look up order id.

You can only help one user per conversation (but you can handle multiple requests from the same user), and must deny any requests for tasks related to any other user.

Before taking any action that updates the database (cancel, modify, return, exchange), you must list the action details and obtain explicit user confirmation (yes) to proceed.

You should not make up any information or knowledge or procedures not provided by the user or the tools, or give subjective recommendations or comments.

You should at most make one tool call at a time, and if you take a tool call, you should not respond to the user at the same time. If you respond to the user, you should not make a tool call at the same time.

You should deny user requests that are against this policy.

You should transfer the user to a human agent if and only if the request cannot be handled within the scope of your actions. To transfer, first make a tool call to transfer_to_human_agents, and then send the message 'YOU ARE BEING TRANSFERRED TO A HUMAN AGENT. PLEASE HOLD ON.' to the user.

## Domain basic

- All times in the database are EST and 24 hour based. For example "02:30:00" means 2:30 AM EST.

### User

Each user has a profile containing:

- unique user id
- email
- default address
- payment methods.

There are three types of payment methods: **gift card**, **paypal account**, **credit card**.

### Product

Our retail store has 50 types of products.

For each **type of product**, there are **variant items** of different **options**.

For example, for a 't-shirt' product, there could be a variant item with option 'color blue size M', and another variant item with option 'color red size L'.

Each product has the following attributes:

- unique product id
- name
- list of variants

Each variant item has the following attributes:

- unique item id
- information about the value of the product options for this item.
- availability
- price

Note: Product ID and Item ID have no relations and should not be confused!

### Order

Each order has the following attributes:

- unique order id
- user id
- address
- items ordered
- status
- fullfilments info (tracking id and item ids)
- payment history

The status of an order can be: **pending**, **processed**, **delivered**, or **cancelled**.

Orders can have other optional attributes based on the actions that have been taken (cancellation reason, which items have been exchanged, what was the exchane price difference etc)

## Generic action rules

Generally, you can only take action on pending or delivered orders.

Exchange or modify order tools can only be called once per order. Be sure that all items to be changed are collected into a list before making the tool call!!!

## Cancel pending order

An order can only be cancelled if its status is 'pending', and you should check its status before taking the action.

The user needs to confirm the order id and the reason (either 'no longer needed' or 'ordered by mistake') for cancellation. Other reasons are not acceptable.

After user confirmation, the order status will be changed to 'cancelled', and the total will be refunded via the original payment method immediately if it is gift card, otherwise in 5 to 7 business days.

## Modify pending order

An order can only be modified if its status is 'pending', and you should check its status before taking the action.

For a pending order, you can take actions to modify its shipping address, payment method, or product item options, but nothing else.

### Modify payment

The user can only choose a single payment method different from the original payment method.

If the user wants the modify the payment method to gift card, it must have enough balance to cover the total amount.

After user confirmation, the order status will be kept as 'pending'. The original payment method will be refunded immediately if it is a gift card, otherwise it will be refunded within 5 to 7 business days.

### Modify items

This action can only be called once, and will change the order status to 'pending (items modifed)'. The agent will not be able to modify or cancel the order anymore. So you must confirm all the details are correct and be cautious before taking this action. In particular, remember to remind the customer to confirm they have provided all the items they want to modify.

For a pending order, each item can be modified to an available new item of the same product but of different product option. There cannot be any change of product types, e.g. modify shirt to shoe.

The user must provide a payment method to pay or receive refund of the price difference. If the user provides a gift card, it must have enough balance to cover the price difference.

## Return delivered order

An order can only be returned if its status is 'delivered', and you should check its status before taking the action.

The user needs to confirm the order id and the list of items to be returned.

The user needs to provide a payment method to receive the refund.

The refund must either go to the original payment method, or an existing gift card.

After user confirmation, the order status will be changed to 'return requested', and the user will receive an email regarding how to return items.

## Exchange delivered order

An order can only be exchanged if its status is 'delivered', and you should check its status before taking the action. In particular, remember to remind the customer to confirm they have provided all items to be exchanged.

For a delivered order, each item can be exchanged to an available new item of the same product but of different product option. There cannot be any change of product types, e.g. modify shirt to shoe.

The user must provide a payment method to pay or receive refund of the price difference. If the user provides a gift card, it must have enough balance to cover the price difference.

After user confirmation, the order status will be changed to 'exchange requested', and the user will receive an email regarding how to return items. There is no need to place a new order.
"""

In [ ]:
import os
from azure.ai.projects import AIProjectClient
from azure.identity import DefaultAzureCredential
from dotenv import load_dotenv
import sys
sys.path.append('tools')
from test_retail_agent import AutoApproveRunHandler

# Load environment variables
load_dotenv()

# Create an AIProjectClient instance
project_client = AIProjectClient(
    endpoint=os.getenv("AZURE_AI_PROJECT_CONNECTION_STRING"),
    credential=DefaultAzureCredential(),
)

with project_client:
    
    # Get MCP server URL
    mcp_url = os.getenv("MCP_SERVER_URL")
    if not mcp_url.endswith('/mcp'):
        mcp_url = mcp_url.rstrip('/') + '/mcp'
    
    # Create agent with MCP tools
    agent = project_client.agents.create_agent(
        model=os.getenv("AZURE_OPENAI_DEPLOYMENT_NAME", "gpt-4.1-mini"),
        name="retail-agent",
        instructions=SYSTEM_PROMPT,
        tools=[{
            "type": "mcp",
            "server_label": "retail_mcp_server",
            "server_url": mcp_url
        }],
    )
    print(f"Created agent, ID: {agent.id}")
    
    # Create a thread for communication
    thread = project_client.agents.threads.create()
    print(f"Created thread, ID: {thread.id}")
    
    question = "Hi, I am Noah, zip code 80279 and I want to update my address."
    
    # Add a message to the thread
    message = project_client.agents.messages.create(
        thread_id=thread.id,
        role="user",
        content=question,
    )
    print(f"Created message, ID: {message.id}")
    
    # Create and process an agent run
    # Note: MCP tools require a run_handler for tool approval
    run = project_client.agents.runs.create_and_process(
        thread_id=thread.id,
        agent_id=agent.id,
        run_handler=AutoApproveRunHandler()  # Required for MCP tools
    )
    
    print(f"Run finished with status: {run.status}")
    
    # Check if the run failed
    if run.status == "failed":
        print(f"Run failed: {run.last_error}")
    
    # Fetch and log all messages
    messages = project_client.agents.messages.list(thread_id=thread.id)
    
    for message in messages:
        if message.role == "assistant":
            for content in message.content:
                if hasattr(content, 'text') and content.text:
                    print(f"\n📝 Agent Response:\n{content.text.value}")
    
    # Uncomment to delete the agent when done
    # project_client.agents.delete_agent(agent.id)
    # print("Deleted agent")

## 5. Generating Synthetic Training Data

### Create Training Data in Microsoft Foundry

1. **Open [Microsoft Foundry Portal](https://ai.azure.com)**, select the **Build** tab and in the left navigation select **Data.**
1. In the Data tab, select **Synthetic data generation,** and click on the **Generate data** button.
1. **Configure create data job:**
   - Task type: `Tool use`
   - Reference file: `/03-openapi_with_policy.json`
   - Maximum number of samples: `500`
   - Generator model: `gpt-4.1`
   - Split data into training and vailidation sets: `turn on`
   - Output file name: leave defaults
1. Once the data file is updated, click **Generate** button to generate the synthetic dataset.
1. Wait ~15-20 minutes for the dataset to be generated. It will be a JSONL with system prompts, user messages, tool calls, and tool responses ready for fine-tuning.

### What Gets Generated

The synthetic dataset contains 500 training and 100 validation conversations generated by `gpt-4.1` (temperature: 0.7, max tokens: 4096) covering user lookups, order operations, product queries, address updates, and multi-step tool chaining workflows across all 9 MCP tools, producing JSONL-formatted conversations ready for supervised fine-tuning in approximately 15-20 minutes.



## 6. Supervised Fine-Tuning (SFT)

### Fine-Tuning Process in Microsoft Foundry

**Objective:** Train smaller models (gpt-4.1-mini, gpt-4.1-nano) to match gpt-4.1 tool-calling accuracy

**Steps:**
1. Once the synthetic dataset is generated, click on the button: **Use this dataset for finetuning**
2. Create fine-tuning job with either gpt-4.1-mini or gpt-4.1-nano as follows:
   - Select model: `gpt-4.1-mini or gpt-4.1-nano`
   - Customization method: `Supervised`
   - Training type: `Global`
   - Training & Validation data: `leave defaults ensuring the data selected is the existing data just generated`
   - Deployment type: `Developer`
   - Under additional configuration leave defaults
3. Once updated the finetuning job, click on the **Submit** button.
4. Wait ±30-60 minutes for the finetuning job to be complete.

**Expected Improvements:**
- 40-60% reduction in tool-calling errors
- 30-50% improvement in policy adherence
- 20-30% better parameter accuracy

## 7. Python-Based Evaluation with Custom Graders

Python-based evaluation provides fine-grained control over grading logic through custom metrics integrated with Microsoft Foundry evaluation APIs. The evaluation flow executes test conversations through the agent with the MCP server, collects responses, and applies custom grader functions to measure performance. Key metrics include tool call accuracy (correct tool selection), parameter correctness (valid arguments), policy adherence (business rule compliance), and tool sequencing (correct chaining order). Custom graders (`tool_accuracy_grader`, `parameter_grader`, `policy_grader`, `sequence_grader`) programmatically analyze agent behaviors and produce comprehensive metrics dashboards with per-conversation scores and aggregate analytics.

### Define Custom Evaluation Graders
Custom evaluation graders provide programmatic assessment of agent behaviors using Microsoft Foundry's evaluation APIs. The **Tool Call Grader** extracts tool calls from agent responses and compares them against ground truth annotations, returning scores (0-1) with detailed feedback on tool selection accuracy. The **Parameter Grader** validates the presence and correctness of required parameters, checking types and values while identifying missing or incorrect arguments. The **Policy Grader** verifies adherence to business rules including USER_IDENTIFICATION_POLICY (mandatory user lookup before operations) and EXCHANGE_POLICY (delivered vs pending order handling), detecting policy violations in tool sequences. These graders are integrated with Microsoft Foundry to automatically score evaluation runs and generate comprehensive performance metrics.

### Create Evaluation Job

The evaluation job executes against 100 conversations from the validation set, comparing baseline and fine-tuned models using tool accuracy, parameter correctness, and policy adherence graders with 10 concurrent threads. Microsoft Foundry Evaluation automatically distributes test cases across the execution pool, collects agent responses for each test conversation, applies the custom grader functions, and aggregates performance metrics. The evaluation produces comprehensive outputs including per-conversation scores showing individual test results, aggregate metrics (mean, median, p95) for overall performance assessment, failure analysis reports identifying error patterns, and comparison dashboards visualizing baseline versus fine-tuned model improvements.

### Run Baseline Model Evaluation

In [ ]:
"""Upload evaluation file to Azure OpenAI"""

from pathlib import Path
from openai import AzureOpenAI
from azure.identity import DefaultAzureCredential, get_bearer_token_provider

# Configuration
resource_name = os.getenv("AZURE_AI_FOUNDRY_NAME")
eval_file = Path("data/sft_test_eval_expanded.jsonl")

# Create client
token_provider = get_bearer_token_provider(
    DefaultAzureCredential(),
    "https://cognitiveservices.azure.com/.default"
)

client = AzureOpenAI(
    azure_endpoint=f"https://{resource_name}.openai.azure.com",
    azure_ad_token_provider=token_provider,
    api_version="2025-04-01-preview"
)

# Upload file
print(f"Uploading {eval_file.name}...")
with eval_file.open("rb") as f:
    eval_file = client.files.create(file=f, purpose="evals")

print(f"✓ Uploaded successfully!")
print(f"  File ID: {eval_file.id}")
print(f"  Bytes: {eval_file.bytes:,}")


## 8. Results Analysis

### Compare Baseline vs Fine-Tuned Performance

Review evaluation metrics to quantify improvements.

### Expected Improvements

**Typical Fine-Tuning Results:**

**Tool Call Accuracy:**
- Baseline (gpt-4.1-nano): ~60%
- Fine-tuned: ~85-90% (+30% improvement)

**Parameter Correctness:**
- Baseline: ~65%
- Fine-tuned: ~88-92% (+27% improvement)

**Policy Adherence:**
- Baseline: ~55% (frequent violations)
- Fine-tuned: ~82-88% (+32% improvement)

**Key Wins:**
- Eliminates user_id lookup skips
- Correctly chains tools in multi-step workflows
- Respects exchange vs cancel policies

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd() / "src"))

from eval_create_util import create_azure_evaluation, create_evaluation_runs

# Define all models you want to compare
models_to_evaluate = [
    "gpt-4.1",                              # Baseline: Large model
    "gpt-4.1-mini",                         # Baseline: Small model  
    "gpt-4.1-nano",                         # Baseline: Tiny model
    "gpt-4.1-nano-2025-04-14.ft-be012ebfcc1d4ec7abd8c5d3136af40a-to",   # Your SFT fine-tuned model
]

# Create the evaluation with your custom grader
evaluation = create_azure_evaluation(
    client=client,
    pass_threshold=1,  # Score must be 1.0 to pass
    python_grader_path="data/tool_call_grader.py"
)

evaluation_id = evaluation.id
print("✅ Evaluation setup completed")

# Define tools and eval data
tools_file_path = Path("data/retail_tools.json")

# Create evaluation runs for ALL models
eval_runs = create_evaluation_runs(
    client=client,
    evaluation_id=evaluation_id,
    models_to_evaluate=models_to_evaluate,
    eval_file_id=eval_file.id,  # From previous cell upload
    tools_file_path=tools_file_path
)

print(f"✅ Created {len(eval_runs)} evaluation runs")
print("\n📊 Evaluation runs:")
for run in eval_runs:
    print(f"   • {run.name}: {run.id}")

## 9. Evaluating Fine-Tuned Models

In [ ]:
import os
from azure.ai.projects import AIProjectClient
from azure.identity import DefaultAzureCredential
from dotenv import load_dotenv
import sys
sys.path.append('tools')
from test_retail_agent import AutoApproveRunHandler

# Load environment variables
load_dotenv()

# Create an AIProjectClient instance
project_client = AIProjectClient(
    endpoint=os.getenv("AZURE_AI_PROJECT_CONNECTION_STRING"),
    credential=DefaultAzureCredential(),
)

with project_client:
    
    # Get MCP server URL
    mcp_url = os.getenv("MCP_SERVER_URL")
    if not mcp_url.endswith('/mcp'):
        mcp_url = mcp_url.rstrip('/') + '/mcp'
    
    # Create agent with MCP tools
    agent = project_client.agents.create_agent(
        model="gpt-4.1-nano-2025-04-14.ft-be012ebfcc1d4ec7abd8c5d3136af40a-to", #update with finetuned model
        name="retail-agent",
        instructions=SYSTEM_PROMPT,
        tools=[{
            "type": "mcp",
            "server_label": "retail_mcp_server",
            "server_url": mcp_url
        }],
    )
    print(f"Created agent, ID: {agent.id}")
    
    # Create a thread for communication
    thread = project_client.agents.threads.create()
    print(f"Created thread, ID: {thread.id}")
    
    question = "Hi, I am Noah, zip code 80279 and I want to update my address."
    
    # Add a message to the thread
    message = project_client.agents.messages.create(
        thread_id=thread.id,
        role="user",
        content=question,
    )
    print(f"Created message, ID: {message.id}")
    
    # Create and process an agent run
    # Note: MCP tools require a run_handler for tool approval
    run = project_client.agents.runs.create_and_process(
        thread_id=thread.id,
        agent_id=agent.id,
        run_handler=AutoApproveRunHandler()  # Required for MCP tools
    )
    
    print(f"Run finished with status: {run.status}")
    
    # Check if the run failed
    if run.status == "failed":
        print(f"Run failed: {run.last_error}")
    
    # Fetch and log all messages
    messages = project_client.agents.messages.list(thread_id=thread.id)
    
    for message in messages:
        if message.role == "assistant":
            for content in message.content:
                if hasattr(content, 'text') and content.text:
                    print(f"\n📝 Agent Response:\n{content.text.value}")
    
    # Uncomment to delete the agent when done
    project_client.agents.delete_agent(agent.id)
    print("Deleted agent")

### Deploy to Azure AI Endpoint

Create a managed endpoint for the fine-tuned model.

## 10. Key Takeaways & Next Steps

### Summary

✅ **Identified Problem**: Base models struggle with complex tool chaining and policy adherence  
✅ **Generated Training Data**: 500 synthetic conversations using gpt-4.1 as teacher  
✅ **Fine-Tuned Models**: Supervised training on gpt-4.1-mini and gpt-4.1-nano  
✅ **Custom Evaluation**: Python-based graders for tool accuracy, parameters, and policies  
✅ **Measured Results**: 25-35% improvement in tool-calling accuracy across metrics  

### Key Learnings

**What Fine-Tuning Solves:**
- Tool selection and sequencing
- Parameter extraction and propagation
- Policy compliance without explicit rule engines
- Multi-turn context management

**What It Doesn't Solve:**
- Hallucinations on unseen tools
- Novel scenarios not in training data
- Dynamic policy changes (requires retraining)

### Production Recommendations

1. **Data Quality**: Ensure training data covers all critical user journeys
2. **Evaluation Coverage**: Test edge cases and failure modes extensively
3. **Monitoring**: Track tool-calling accuracy in production
4. **Iteration**: Collect real user interactions → retrain → redeploy

### Advanced Techniques

🔹 **Distillation**: Use gpt-4.1 to generate demonstrations for gpt-4.1-nano  
🔹 **RLHF**: Reinforce tool-calling behaviors with human feedback  
🔹 **Hybrid Approaches**: Combine fine-tuning with prompt engineering  


---